# Proyecto del equipo: <título del proyecto>

**Equipo:** <nombre del equipo> · **Integrantes:** <nombre 1>, <nombre 2>, <nombre 3>

Este notebook es **el proyecto del equipo**. Se llena por partes, una sección en cada clase, y al final es la base de la sustentación.

| Sección | Se trabaja en | Fase de CRISP-DM |
|---|---|---|
| 0. Configuración | 26 de octubre | |
| 1. El problema | 26 de octubre | 1. Comprensión del negocio |
| 2. Los datos | 26 de octubre | 2. Comprensión de los datos |
| 3. Limpieza | 27 de octubre | 3. Preparación de los datos |
| 4. Análisis exploratorio (EDA) | 27 de octubre | 2 y 3 |
| 5. Modelo | 28 de octubre | 4. Modelado y 5. Evaluación |
| 6. Conclusiones | 11 de noviembre | 5. Evaluación y 6. Despliegue |

**Cómo usar la plantilla**

- Las celdas y tablas marcadas con **COMPLETAR** son las que el equipo debe cambiar.
- Las celdas que dicen **Su código aquí** son espacios libres: ahí el equipo explora a su manera, con las preguntas y los consejos de cada sección como guía.
- Las demás celdas ya funcionan: léanlas, ejecútenlas y entiendan qué hacen. Pueden modificarlas si su caso lo necesita.
- Ejecuten el notebook **de arriba hacia abajo**. Antes de cada commit, usen *Restart Kernel and Run All* para comprobar que corre completo.
- Al terminar cada sesión: `git pull`, `git add notebooks/proyecto_equipo.ipynb`, `git commit -m "feat: ..."` y `git push`.

## 0. Configuración

**COMPLETAR una sola vez.** Estas cinco variables son lo único que necesita el resto del notebook para funcionar con su dataset.

- `ARCHIVO`: ruta al CSV, **relativa a la carpeta `notebooks/`**. Por eso empieza con `../`, que significa "subir una carpeta".
- `SEPARADOR`: el carácter que separa las columnas. Casi siempre es `","`; los archivos de datos.gov.co suelen usar `";"`.
- `OBJETIVO`: el nombre exacto de la columna que quieren predecir o explicar (la variable objetivo o *target*).
- `TIPO_PROBLEMA`: `"regresion"` si el objetivo es un número (precio, ventas, días) o `"clasificacion"` si es una categoría (sí/no, tipo de cliente).
- `COLUMNAS_A_EXCLUIR`: columnas que **no** deben usarse para predecir, por ejemplo identificadores o columnas que solo se conocen después del resultado (*data leakage*).

In [ ]:
ARCHIVO = "../data/raw/COMPLETAR.csv"
SEPARADOR = ","
OBJETIVO = "COMPLETAR"
TIPO_PROBLEMA = "regresion"  # "regresion" o "clasificacion"
COLUMNAS_A_EXCLUIR = []  # por ejemplo: ["id", "fecha_cancelacion"]

SEMILLA = 42  # fija el azar para que los resultados se repitan

In [ ]:
# Librerías. Si alguna falla, instálenla con:  uv add <libreria>
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

assert TIPO_PROBLEMA in ("regresion", "clasificacion"), "TIPO_PROBLEMA debe ser 'regresion' o 'clasificacion'"
print("Configuración lista. Archivo:", ARCHIVO)

---
## 1. El problema (26 de octubre)

**COMPLETAR.** Antes de tocar los datos, el equipo debe ponerse de acuerdo en **qué problema resuelve** y **cómo sabrá que lo logró**. Respondan en la tabla (doble clic sobre esta celda para editarla).

| Pregunta | Respuesta del equipo |
|---|---|
| ¿Qué **decisión** de negocio se quiere mejorar? | |
| ¿**Quién** toma esa decisión? | |
| ¿Qué **tipo de análisis** es el principal? (descriptivo, diagnóstico, predictivo o prescriptivo) | |
| ¿Cuál es la **variable objetivo**? | |
| **Criterio de éxito:** ¿qué métrica y qué umbral indican que el proyecto funcionó? | |
| **KPI 1** (SMART: específico, medible, alcanzable, relevante y con periodo) | |
| **KPI 2** (SMART) | |

> **Ejemplo (inmobiliaria):** decisión: fijar el precio de publicación de una casa · quién: el asesor comercial · tipo: predictivo · objetivo: `SalePrice` · éxito: error porcentual medio (MAPE) menor al 15 % · KPI 1: precio mediano por m² por barrio, cada trimestre · KPI 2: días promedio que tarda una casa en venderse, cada mes.

---
## 2. Los datos (26 de octubre)

### 2.1 Cargar y mirar

`pd.read_csv` lee el archivo y lo convierte en un **DataFrame**: una tabla con filas y columnas.

In [ ]:
df = pd.read_csv(ARCHIVO, sep=SEPARADOR)

print(f"El dataset tiene {df.shape[0]:,} filas y {df.shape[1]} columnas.")
df.head()

**COMPLETAR.** ¿Qué representa **una fila**? (por ejemplo: "una venta de una casa", "un cliente en un mes", "un viaje").

> Respuesta:

### 2.2 Exploren los datos

**Esta parte es libre.** Añadan las celdas de código que necesiten para el análisis y puedan responder estas preguntas. No hay una única forma correcta de hacerlo.

| Pregunta | Por qué importa |
|---|---|
| ¿Qué tipo tiene cada columna? ¿Alguna columna de números aparece como texto? | Un número leído como texto (`"1.500,00"`, `"$200"`, `"N/A"`) no se puede sumar ni promediar |
| ¿Qué columnas tienen datos faltantes? ¿Cuántos? | Mañana decidirán qué hacer con ellos, y primero hay que saber **por qué** faltan |
| ¿Hay filas repetidas? | Una venta contada dos veces infla los resultados |
| ¿Hay columnas con un valor distinto en cada fila? | Suelen ser identificadores (`id`, cédula): no sirven para predecir |
| ¿Cómo se ve la variable objetivo? | En regresión, ¿hay valores extremos? En clasificación, ¿hay muchos más casos de una clase que de otra? |
| ¿Hay valores imposibles? | Edades negativas, precios en cero, fechas en el futuro |
| ¿Hay columnas que solo se conocen **después** del resultado? | Son *data leakage*: van en `COLUMNAS_A_EXCLUIR` |

**Herramientas útiles** (copien la que necesiten en una celda de código):

```python
df.info()                                   # tipos de cada columna y cuántos valores no nulos tiene
df.describe()                               # resumen de las columnas numéricas: mínimo, máximo, promedio, cuartiles
df.describe(exclude="number")               # resumen de las columnas de texto: valores únicos y el más frecuente
df.isna().sum().sort_values(ascending=False)  # nulos por columna, de mayor a menor
df.isna().mean().mul(100).round(1)          # % de nulos por columna
df.duplicated().sum()                       # cuántas filas repetidas hay
df.nunique()                                # cuántos valores distintos tiene cada columna
df["columna"].value_counts()                # cuántas veces aparece cada categoría
df[OBJETIVO].describe()                     # resumen de la variable objetivo
sns.histplot(df[OBJETIVO])                  # distribución de una variable numérica
sns.countplot(x=df[OBJETIVO])               # casos por clase de una variable categórica
```

**Recomendaciones**

- Una pregunta por celda, y debajo una línea de texto con lo que encontraron. Así el notebook se lee como un informe.
- Tengan a mano el **diccionario de datos** del dataset: muchas dudas ("¿qué significa este código?", "¿por qué está vacío?") se responden ahí.
- Si una salida es muy larga, muestren solo una parte: `.head(10)`.
- Si algo les llama la atención, anótenlo aunque todavía no sepan qué hacer con ello: irá a la tabla de la sección 2.3.

In [1]:
# Su código aquí

In [2]:
# Su código aquí

In [3]:
# Su código aquí

### 2.3 Problemas de calidad e hipótesis

**COMPLETAR** con lo que encontraron en la exploración. La limpieza de la próxima clase atacará estos problemas, y el EDA pondrá a prueba las hipótesis.

| # | Problema de calidad encontrado | Columna(s) |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |

| # | Hipótesis: ¿qué creemos que explica la variable objetivo? |
|---|---|
| H1 | |
| H2 | |
| H3 | |

---
## 3. Limpieza (27 de octubre)

Trabajamos sobre una **copia** (`df_limpio`) para no perder los datos originales. Cada decisión se escribe en una celda y se justifica en el registro de abajo.

Antes de rellenar un nulo, pregúntense **por qué falta el dato**:

| Si el dato falta porque... | Decisión típica | Código |
|---|---|---|
| **No existe** (la casa no tiene piscina) | Poner un valor que lo diga | `df_limpio["col"] = df_limpio["col"].fillna("No tiene")` |
| **No se registró** (falla de captura) | Imputar con la mediana (números) o la moda (categorías) | `df_limpio["col"] = df_limpio["col"].fillna(df_limpio["col"].median())` |
| Son **muy pocas filas** con problemas | Eliminar esas filas | `df_limpio = df_limpio.dropna(subset=["col"])` |
| La columna tiene **casi todo vacío** y no aporta | Eliminar la columna | `df_limpio = df_limpio.drop(columns=["col"])` |

Otras operaciones frecuentes:

```python
df_limpio = df_limpio.drop_duplicates()                                  # quitar filas repetidas
df_limpio = df_limpio[df_limpio["precio"] > 0]                            # quitar valores imposibles
df_limpio["fecha"] = pd.to_datetime(df_limpio["fecha"], errors="coerce")  # texto -> fecha
df_limpio["valor"] = pd.to_numeric(df_limpio["valor"].str.replace(",", "."), errors="coerce")  # "1,5" -> 1.5
df_limpio["ciudad"] = df_limpio["ciudad"].str.strip().str.lower()        # unificar textos
```

In [ ]:
df_limpio = df.copy()

# Las filas sin variable objetivo no sirven para entrenar ni para evaluar.
df_limpio = df_limpio.dropna(subset=[OBJETIVO])

# COMPLETAR: decisión 1


# COMPLETAR: decisión 2


# COMPLETAR: decisión 3


print(f"Filas: {len(df):,} antes y {len(df_limpio):,} después de la limpieza")

In [ ]:
# Comprobación: columnas que todavía tienen nulos
nulos_restantes = df_limpio.isna().sum()
nulos_restantes = nulos_restantes[nulos_restantes > 0].sort_values(ascending=False)
if nulos_restantes.empty:
    print("No quedan nulos.")
else:
    print("Columnas con nulos después de la limpieza:")
    print(nulos_restantes)
    print("\nLos nulos que queden en las variables del modelo se rellenan automáticamente en la sección 5.")

**COMPLETAR: registro de decisiones.** Es lo que se muestra en la sustentación cuando pregunten "¿qué hicieron con los datos?".

| # | Problema | Decisión | Por qué (criterio de negocio) |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

In [ ]:
# Guardar los datos limpios. La carpeta data/processed/ está en .gitignore:
# no se sube a GitHub porque se puede regenerar ejecutando este notebook.
Path("../data/processed").mkdir(parents=True, exist_ok=True)
df_limpio.to_csv("../data/processed/datos_limpios.csv", index=False)
print("Guardado en data/processed/datos_limpios.csv")

---
## 4. Análisis exploratorio, EDA (27 de octubre)

El EDA no es hacer muchas gráficas: es **responder preguntas** con evidencia. Al final de la sección deben quedar **3 hallazgos**.

### 4.1 ¿Qué variables se relacionan más con el objetivo?

- **Regresión:** la **correlación** va de -1 a 1. Cerca de 1, cuando una sube la otra también; cerca de -1, cuando una sube la otra baja; cerca de 0, no hay relación *lineal*.
- **Clasificación:** comparamos el **promedio** de cada variable numérica en cada clase. Si los promedios son muy distintos, la variable ayuda a separar las clases.

In [ ]:
# Numéricas sin el objetivo y sin identificadores (columnas donde todos los valores son distintos)
numericas = [
    c for c in df_limpio.select_dtypes(include="number").columns
    if c != OBJETIVO and c not in COLUMNAS_A_EXCLUIR and df_limpio[c].nunique() < len(df_limpio)
]

if TIPO_PROBLEMA == "regresion":
    relacion = df_limpio[numericas].corrwith(df_limpio[OBJETIVO]).dropna()
    relacion = relacion.reindex(relacion.abs().sort_values(ascending=False).index)
    top = relacion.head(10)
    fig, ax = plt.subplots(figsize=(8, 4))
    top.sort_values().plot.barh(ax=ax, color=np.where(top.sort_values() > 0, "#2A9D8F", "#C8553D"))
    ax.set_title(f"Correlación con {OBJETIVO} (las 10 más fuertes)")
    plt.show()
else:
    relacion = df_limpio.groupby(OBJETIVO)[numericas].mean().T
    print("Promedio de cada variable numérica por clase:")
relacion.head(10)

### 4.2 Una variable numérica frente al objetivo

**COMPLETAR:** elijan una variable numérica de la tabla anterior y escriban su nombre en `VARIABLE_NUMERICA`.

In [ ]:
VARIABLE_NUMERICA = numericas[0] if len(numericas) else None  # COMPLETAR: por ejemplo "area"

fig, ax = plt.subplots(figsize=(8, 4))
if TIPO_PROBLEMA == "regresion":
    sns.scatterplot(data=df_limpio, x=VARIABLE_NUMERICA, y=OBJETIVO, alpha=0.4, ax=ax)
else:
    sns.boxplot(data=df_limpio, x=OBJETIVO, y=VARIABLE_NUMERICA, ax=ax)
ax.set_title(f"{VARIABLE_NUMERICA} frente a {OBJETIVO}")
plt.show()

### 4.3 Una variable categórica frente al objetivo

**COMPLETAR:** elijan una variable categórica (texto o pocas categorías) y escriban su nombre en `VARIABLE_CATEGORICA`. La gráfica muestra las 10 categorías con más filas.

In [ ]:
categoricas = df_limpio.select_dtypes(exclude=["number", "datetime"]).columns.drop(OBJETIVO, errors="ignore")
VARIABLE_CATEGORICA = categoricas[0] if len(categoricas) else None  # COMPLETAR: por ejemplo "barrio"

if VARIABLE_CATEGORICA is None:
    print("El dataset no tiene variables categóricas.")
else:
    principales = df_limpio[VARIABLE_CATEGORICA].value_counts().head(10).index
    datos = df_limpio[df_limpio[VARIABLE_CATEGORICA].isin(principales)]
    fig, ax = plt.subplots(figsize=(9, 4))
    if TIPO_PROBLEMA == "regresion":
        orden = datos.groupby(VARIABLE_CATEGORICA)[OBJETIVO].median().sort_values().index
        sns.boxplot(data=datos, x=VARIABLE_CATEGORICA, y=OBJETIVO, order=orden, ax=ax)
    else:
        pd.crosstab(datos[VARIABLE_CATEGORICA], datos[OBJETIVO], normalize="index").plot.bar(stacked=True, ax=ax)
        ax.set_ylabel("proporción")
    ax.set_title(f"{OBJETIVO} según {VARIABLE_CATEGORICA}")
    ax.tick_params(axis="x", rotation=45)
    plt.show()

### 4.4 Un KPI calculado con los datos

Los KPIs de la sección 1 se calculan casi siempre con `groupby`: **agrupar** por una columna y **resumir** otra. Ajusten la celda para calcular uno de sus KPIs (por ejemplo, la mediana del objetivo por categoría, o el total por mes).

In [ ]:
if VARIABLE_CATEGORICA is not None:
    if TIPO_PROBLEMA == "regresion":
        kpi = df_limpio.groupby(VARIABLE_CATEGORICA)[OBJETIVO].agg(casos="count", mediana="median", promedio="mean")
    else:
        kpi = pd.crosstab(df_limpio[VARIABLE_CATEGORICA], df_limpio[OBJETIVO], margins=True, margins_name="Total")
    display(kpi.sort_values(kpi.columns[0], ascending=False).head(10))

### 4.5 Hallazgos del EDA

**COMPLETAR.** Un hallazgo tiene tres partes: **qué** encontraron, **con qué evidencia** (gráfica o número) y **qué implica** para el modelo o para el negocio. Indiquen también si confirma o rechaza alguna hipótesis de la sección 2.3.

| # | Hallazgo | Evidencia | Implicación | Hipótesis |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

> **Ejemplo:** "Las casas con calidad 9-10 cuestan casi el triple que las de calidad 5" · evidencia: boxplot de precio por calidad · implicación: incluir la calidad en el modelo · confirma H1.

---
## 5. Modelo (28 de octubre)

El objetivo de esta sección **no** es encontrar el mejor modelo posible, sino responder dos preguntas:

1. ¿Nuestro modelo le gana a un ***baseline***, es decir, a una regla simple sin inteligencia?
2. ¿Cumple el **criterio de éxito** que definimos en la sección 1?

### 5.1 Variables del modelo

Si dejan `VARIABLES = []`, el notebook usa automáticamente las columnas numéricas y las categóricas con pocas categorías (hasta 15), sin el objetivo, sin identificadores y sin las columnas de `COLUMNAS_A_EXCLUIR`. **COMPLETAR (recomendado):** escriban la lista con las variables que el EDA mostró como importantes.

In [ ]:
VARIABLES = []  # COMPLETAR (opcional): por ejemplo ["area", "habitaciones", "barrio"]

if not VARIABLES:
    candidatas = df_limpio.columns.drop([OBJETIVO, *COLUMNAS_A_EXCLUIR], errors="ignore")
    VARIABLES = [
        c for c in candidatas
        if df_limpio[c].nunique() < len(df_limpio)  # descarta identificadores
        and (pd.api.types.is_numeric_dtype(df_limpio[c]) or df_limpio[c].nunique() <= 15)
    ]

X = df_limpio[VARIABLES]
y = df_limpio[OBJETIVO]
vars_numericas = X.select_dtypes(include="number").columns.tolist()
vars_categoricas = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(vars_numericas)} variables numéricas: {vars_numericas}")
print(f"{len(vars_categoricas)} variables categóricas: {vars_categoricas}")

### 5.2 Separar entrenamiento y prueba

El modelo **aprende** con el 80 % de los datos (*train*) y lo **evaluamos** con el 20 % restante (*test*), que nunca vio. Así medimos qué tan bien funcionará con datos nuevos, no qué tan bien memorizó.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA,
    stratify=y if TIPO_PROBLEMA == "clasificacion" else None,  # misma proporción de clases en train y test
)
print(f"Train: {len(X_train):,} filas · Test: {len(X_test):,} filas")

### 5.3 Preparar las variables automáticamente

Los modelos solo entienden números y no aceptan nulos. Este bloque prepara los datos **aprendiendo solo de train**, lo que evita la fuga de información (*data leakage*):

- **Numéricas:** rellena los nulos con la mediana y las pone en la misma escala.
- **Categóricas:** rellena los nulos con la categoría más frecuente y crea una columna 0/1 por categoría (*one-hot encoding*).

En la clase del 11 de noviembre veremos en detalle cómo funcionan estos `Pipeline`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocesamiento = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), vars_numericas),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), vars_categoricas),
])

### 5.4 Entrenar el *baseline* y dos modelos

| Modelo | Regresión | Clasificación |
|---|---|---|
| ***Baseline*** | Predice siempre la **mediana** | Predice siempre la **clase más frecuente** |
| **Modelo lineal** | Regresión lineal | Regresión logística |
| **Árbol de decisión** | Árbol con profundidad máxima 5 | Árbol con profundidad máxima 5 |

Cada modelo se une a la preparación de la sección 5.3 con `make_pipeline`, se entrena con `fit` y predice con `predict`.

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, mean_absolute_error, mean_absolute_percentage_error,
    precision_score, r2_score, recall_score,
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

if TIPO_PROBLEMA == "regresion":
    modelos = {
        "Baseline (mediana)": DummyRegressor(strategy="median"),
        "Regresión lineal": LinearRegression(),
        "Árbol de decisión": DecisionTreeRegressor(max_depth=5, random_state=SEMILLA),
    }
else:
    modelos = {
        "Baseline (clase más frecuente)": DummyClassifier(strategy="most_frequent"),
        "Regresión logística": LogisticRegression(max_iter=1000),
        "Árbol de decisión": DecisionTreeClassifier(max_depth=5, random_state=SEMILLA),
    }


def evaluar(y_real, y_pred):
    """Calcula las métricas del tipo de problema."""
    if TIPO_PROBLEMA == "regresion":
        return {
            "MAE": mean_absolute_error(y_real, y_pred),
            "MAPE (%)": mean_absolute_percentage_error(y_real, y_pred) * 100,
            "R2": r2_score(y_real, y_pred),
        }
    promedio = "binary" if y.nunique() == 2 else "macro"
    positiva = y.value_counts().index[-1]  # en problemas binarios, la clase menos frecuente
    opciones = {"average": promedio, "zero_division": 0}
    if promedio == "binary":
        opciones["pos_label"] = positiva
    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred, **opciones),
        "recall": recall_score(y_real, y_pred, **opciones),
        "F1": f1_score(y_real, y_pred, **opciones),
    }


entrenados, filas = {}, []
for nombre, modelo in modelos.items():
    pipe = make_pipeline(preprocesamiento, modelo).fit(X_train, y_train)
    entrenados[nombre] = pipe
    filas.append({"modelo": nombre, "conjunto": "train", **evaluar(y_train, pipe.predict(X_train))})
    filas.append({"modelo": nombre, "conjunto": "test", **evaluar(y_test, pipe.predict(X_test))})

resultados = (
    pd.DataFrame(filas)
    .pivot(index="modelo", columns="conjunto")
    .reindex(columns=["train", "test"], level=1)
    .reindex(list(modelos))
    .round(3)
)
resultados

**Cómo leer la tabla**

- **Regresión:** MAE y MAPE son errores (**más bajo es mejor**); MAE está en las unidades del objetivo y MAPE en porcentaje. R² va de 0 a 1 (**más alto es mejor**).
- **Clasificación:** todas las métricas van de 0 a 1 (**más alto es mejor**). Si las clases están desbalanceadas, miren **recall** y **F1**, no solo *accuracy*. En problemas de dos clases, precision y recall se calculan para la clase menos frecuente.
- **Train frente a test:** si el modelo es mucho mejor en train que en test, está **memorizando** (*overfitting*).

> Si `MAPE` sale enorme, revisen si el objetivo tiene valores en cero o muy cercanos a cero: en ese caso usen MAE.

### 5.5 ¿En qué se equivoca el modelo?

Elijan el modelo con mejor resultado en **test** y escriban su nombre en `MEJOR_MODELO`.

In [ ]:
MEJOR_MODELO = list(modelos)[1]  # COMPLETAR: nombre exacto de la tabla
mejor = entrenados[MEJOR_MODELO]
pred_test = mejor.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
if TIPO_PROBLEMA == "regresion":
    ax.scatter(y_test, pred_test, alpha=0.4)
    limites = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
    ax.plot(limites, limites, color="red", linestyle="--", label="predicción perfecta")
    ax.set_xlabel("valor real")
    ax.set_ylabel("valor predicho")
    ax.legend()
else:
    from sklearn.metrics import ConfusionMatrixDisplay
    ConfusionMatrixDisplay.from_predictions(y_test, pred_test, ax=ax, colorbar=False)
ax.set_title(f"{MEJOR_MODELO}: test")
plt.show()

**COMPLETAR: interpretación del modelo.**

| Pregunta | Respuesta del equipo |
|---|---|
| ¿Qué modelo eligieron y por qué? | |
| ¿Cuánto mejora frente al *baseline*? (por ejemplo, "el MAPE baja de 31 % a 12 %") | |
| ¿Cumple el criterio de éxito de la sección 1? | |
| ¿Hay *overfitting*? (compare train y test) | |
| ¿En qué casos se equivoca más? (mire la gráfica) | |

---
## 6. Conclusiones (11 de noviembre)

### 6.1 Guardar el modelo

`joblib` guarda en un archivo el pipeline completo (preparación + modelo), listo para usarse sin volver a entrenar. La carpeta `models/` está en `.gitignore`: el modelo se regenera ejecutando el notebook.

In [ ]:
import joblib

Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(mejor, "../models/modelo.joblib")
print("Modelo guardado en models/modelo.joblib")

# Comprobación: cargarlo y predecir las primeras 3 filas de test
modelo_cargado = joblib.load("../models/modelo.joblib")
print(pd.DataFrame({"real": y_test.head(3).to_numpy(), "predicho": modelo_cargado.predict(X_test.head(3))}))

### 6.2 Conclusiones para el negocio

**COMPLETAR.** Escríbanlas pensando en la persona que toma la decisión, no en un público técnico. Son la base de las diapositivas finales.

1. **Respuesta al problema:** ¿qué puede hacer ahora el negocio que antes no podía?
2. **Resultado principal:** en una frase y en unidades de negocio (por ejemplo, "el avalúo se equivoca en promedio un 11 %").
3. **Límites:** ¿cuándo **no** conviene confiar en el modelo? (rangos de valores, grupos con pocos datos, periodos).
4. **Siguientes pasos:** ¿qué harían con un mes más? (más datos, otras variables, otros modelos, una API o un dashboard).